# Selective Scan Benchmark: GPU/TPU (Colab Compatible)

This Colab notebook benchmarks the `selective_scan_fn` kernel on both GPU and TPU, measuring kernel-level latency for a synthetic input. It automatically detects the available accelerator, installs required dependencies, and reports mean/std latency for 30 runs.

**Sections:**
1. Environment setup and dependency installation
2. Device detection and configuration
3. Import selective_scan_fn and dependencies
4. Generate test input data
5. Run benchmark on GPU
6. Run benchmark on TPU (if available)
7. Results summary and visualization

All code and comments are in English for reproducibility.

## 1. Environment Setup and Dependency Installation

This section installs all required Python packages for running the benchmark on Colab. It ensures torch, torch_xla (for TPU), and mamba-ssm (for selective_scan_fn) are available. If running on GPU, it also installs triton if needed.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Check for TPU
    try:
        import torch_xla
        TPU_AVAILABLE = True
    except ImportError:
        TPU_AVAILABLE = False
    if TPU_AVAILABLE:
        # Use the official Colab torch_xla install script for best compatibility
        # This will install the correct torch and torch_xla versions for Colab TPU
        import os
        os.system('pip install --quiet mamba-ssm')
        os.system('curl -s https://raw.githubusercontent.com/pytorch/xla/main/contrib/scripts/env-setup.py | python -')
    else:
        # For GPU/CPU
        os.system('pip install --quiet torch==2.2.0 triton==2.2.0 mamba-ssm')
else:
    print('Not running in Colab. Please ensure torch, torch_xla (for TPU), and mamba-ssm are installed.')

## 2. Device Detection and Configuration

Detect if the current runtime is using a TPU or GPU, and set up the appropriate device for PyTorch or torch_xla. Print the detected device type and name.

In [ ]:
# Detect device: TPU or GPU
import torch

def detect_device():
    try:
        import torch_xla.core.xla_model as xm
        device = xm.xla_device()
        device_type = 'TPU'
        device_name = 'TPU (torch_xla)'
    except ImportError:
        if torch.cuda.is_available():
            device = torch.device('cuda')
            device_type = 'GPU'
            device_name = torch.cuda.get_device_name(0)
        else:
            device = torch.device('cpu')
            device_type = 'CPU'
            device_name = 'CPU'
    return device, device_type, device_name

device, device_type, device_name = detect_device()
print(f"Detected device: {device_type} - {device_name}")

## 3. Import selective_scan_fn and Dependencies

Import the selective_scan_fn kernel and any required modules. If running on TPU, use a minimal PyTorch XLA-compatible scan; if on GPU, use mamba-ssm's selective_scan_fn.

In [ ]:
# Import selective_scan_fn and dependencies
import numpy as np

if device_type == 'TPU':
    import torch_xla.core.xla_model as xm
    # Define a minimal scan kernel for demonstration (PyTorch XLA does not support mamba-ssm)
    def selective_scan_fn(x):
        # Simulate a scan: cumulative sum along the sequence dimension
        return torch.cumsum(x, dim=-1)
else:
    try:
        from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    except ImportError:
        # Fallback: define a dummy scan for demonstration
        def selective_scan_fn(x):
            return torch.cumsum(x, dim=-1)

print(f"selective_scan_fn ready for device: {device_type}")

## 4. Generate Test Input Data

Create random input tensors for benchmarking: batch=1, seq_len=4096, dim=1024, d_state=16, chunk=512, dtype=FP16. Move the tensor to the detected device.

In [ ]:
# Generate random input tensor for benchmarking
batch = 1
seq_len = 4096
dim = 1024
d_state = 16  # Not used in this minimal demo, but included for completeness
chunk = 512

# Input shape: (batch, dim, seq_len)
x = torch.randn(batch, dim, seq_len, dtype=torch.float16)
x = x.to(device)
print(f"Input tensor shape: {x.shape}, dtype: {x.dtype}, device: {x.device}")

## 5. Run selective_scan_fn Benchmark on GPU

If a GPU is available, run the selective_scan_fn kernel 30 times, record wall-clock latency for each run, and compute mean and standard deviation. Results are printed and saved to summary.json.

In [ ]:
import time
import json

def run_benchmark_gpu(x, n_repeats=30):
    latencies = []
    # Warmup
    for _ in range(3):
        _ = selective_scan_fn(x)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
    for _ in range(n_repeats):
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t0 = time.perf_counter()
        _ = selective_scan_fn(x)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)
    return latencies

if device_type == 'GPU':
    print('Running selective_scan_fn benchmark on GPU...')
    latencies = run_benchmark_gpu(x)
    mean_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    print(f"GPU mean latency: {mean_latency:.4f} ms, std: {std_latency:.4f} ms")
    # Save result
    result = {
        'device': device_name,
        'mean_latency_ms': mean_latency,
        'std_latency_ms': std_latency,
        'latencies_ms': latencies,
        'n_repeats': len(latencies)
    }
    with open('gpu_selective_scan_summary.json', 'w') as f:
        json.dump(result, f, indent=2)
    print('Result saved to gpu_selective_scan_summary.json')

## 6. Run selective_scan_fn Benchmark on TPU

If a TPU is available, run the selective_scan_fn kernel 30 times using torch_xla, record wall-clock latency, and compute mean and standard deviation. Results are printed and saved to summary.json.

In [ ]:
def run_benchmark_tpu(x, n_repeats=30):
    import torch_xla.core.xla_model as xm
    latencies = []
    # Warmup
    for _ in range(3):
        _ = selective_scan_fn(x)
        xm.mark_step()
    for _ in range(n_repeats):
        xm.mark_step()
        t0 = time.perf_counter()
        _ = selective_scan_fn(x)
        xm.mark_step()
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)
    return latencies

if device_type == 'TPU':
    print('Running selective_scan_fn benchmark on TPU...')
    latencies = run_benchmark_tpu(x)
    mean_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    print(f"TPU mean latency: {mean_latency:.4f} ms, std: {std_latency:.4f} ms")
    # Save result
    result = {
        'device': device_name,
        'mean_latency_ms': mean_latency,
        'std_latency_ms': std_latency,
        'latencies_ms': latencies,
        'n_repeats': len(latencies)
    }
    with open('tpu_selective_scan_summary.json', 'w') as f:
        json.dump(result, f, indent=2)
    print('Result saved to tpu_selective_scan_summary.json')

## 7. Results Summary and Visualization

This section loads the saved summary.json files (if present), prints the results, and visualizes the latency distributions for GPU and TPU runs.

In [ ]:
import matplotlib.pyplot as plt
import os

def load_and_print_result(filename, label):
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            result = json.load(f)
        print(f"{label} result:")
        print(json.dumps(result, indent=2))
        return result
    else:
        print(f"{label} result file not found: {filename}")
        return None

gpu_result = load_and_print_result('gpu_selective_scan_summary.json', 'GPU')
tpu_result = load_and_print_result('tpu_selective_scan_summary.json', 'TPU')

# Plot latency distributions if available
plt.figure(figsize=(8, 4))
if gpu_result:
    plt.hist(gpu_result['latencies_ms'], bins=15, alpha=0.6, label='GPU')
if tpu_result:
    plt.hist(tpu_result['latencies_ms'], bins=15, alpha=0.6, label='TPU')
plt.xlabel('Latency (ms)')
plt.ylabel('Count')
plt.title('selective_scan_fn Latency Distribution')
plt.legend()
plt.show()

## 8. (Optional) Download Results

This cell provides a link to download the summary JSON files for local analysis or manuscript integration.

In [ ]:
from google.colab import files
for fname in ['gpu_selective_scan_summary.json', 'tpu_selective_scan_summary.json']:
    if os.path.exists(fname):
        files.download(fname)